# Αποσύνθεση της Μεταβλητότητας Διακανονισμού Αξιώσεων στην Οργανωτική Ιεραρχία μιας Ασφαλιστικής με την PROC NESTED


## Περίληψη

Μια ασφαλιστική εταιρεία περιουσίας και ατυχημάτων θέλει να μάθει **πού**
προέρχεται η ασυνέπεια στους διακανονισμούς αξιώσεων: οφείλεται σε
διαφορές μεταξύ γεωγραφικών περιφερειών, μεταξύ υποκαταστημάτων, μεταξύ
μεμονωμένων πραγματογνωμόνων, ή απλώς σε θόρυβο από αξίωση σε αξίωση;
Αυτό το notebook χτίζει ένα συνθετικό, πλήρως ένθετο βιβλίο δεδομένων
αξιώσεων αυτοκινήτου (Περιφέρεια > Υποκατάστημα > Πραγματογνώμονας >
αξίωση) και χρησιμοποιεί την **PROC NESTED** για να εκτελέσει μια
ανάλυση διακύμανσης τυχαίων επιδράσεων, εκτιμώντας τη συνιστώσα
διακύμανσης που συνεισφέρει κάθε επίπεδο της ιεραρχίας και αναφέροντάς
την ως ποσοστό του συνόλου. Το αποτέλεσμα λέει στη διοίκηση ποιο
οργανωτικό επίπεδο πρέπει να στοχεύσει για βελτιώσεις στη συνέπεια
διακανονισμού.


## Πηγές Δεδομένων

Όλα τα δεδομένα δημιουργούνται εσωτερικά από το πρώτο βήμα DATA — χωρίς
εξωτερικά αρχεία, χωρίς δίκτυο. Ο σχεδιασμός είναι **πλήρως ένθετος**:
κάθε υποκατάστημα ανήκει σε ακριβώς μία περιφέρεια, κάθε πραγματογνώμονας
σε ακριβώς ένα υποκατάστημα, και κάθε αξίωση σε ακριβώς έναν
πραγματογνώμονα.

| Σύνολο Δεδομένων | Γραμμές | Μεταβλητή | Τύπος | Ρόλος | Περιγραφή |
|---------|------|----------|------|------|-------------|
| `claims` | 40 | `Region` | χαρακτήρας | CLASS (επίπεδο 1, εξωτερικό) | Γεωγραφική περιφέρεια (5 περιφέρειες: ΒΑ, ΝΑ, ΜΔ, ΔΑ, ΝΔ) |
| | | `Branch` | χαρακτήρας | CLASS (επίπεδο 2, ένθετο εντός Περιφέρειας) | Κωδικός υποκαταστήματος (2 υποκαταστήματα ανά περιφέρεια) |
| | | `Adjuster` | χαρακτήρας | CLASS (επίπεδο 3, ένθετο εντός Υποκαταστήματος) | Αναγνωριστικό πραγματογνώμονα αξιώσεων (2 πραγματογνώμονες ανά υποκατάστημα) |
| | | `Settlement` | αριθμ. | VAR (εξαρτημένη) | Αποζημίωση που καταβλήθηκε για την αξίωση αυτοκινήτου, σε USD |
| | | `CycleDays` | αριθμ. | VAR (εξαρτημένη) | Ημέρες από την πρώτη γνωστοποίηση της ζημιάς έως τον διακανονισμό |

Συνθετική δομή: 5 περιφέρειες x 2 υποκαταστήματα x 2 πραγματογνώμονες x
2 αξιώσεις = 40 παρατηρήσεις. Μια τυχαία επίδραση περιφέρειας, μια
τυχαία επίδραση υποκαταστήματος-εντός-περιφέρειας, μια τυχαία επίδραση
πραγματογνώμονα-εντός-υποκαταστήματος, και ένα υπόλοιπο επιπέδου
αξίωσης προστίθενται διαδοχικά μέσω της `rand('NORMAL', ...)` ώστε οι
συνιστώσες διακύμανσης να είναι ανακτήσιμες. Η επίδραση περιφέρειας
σχεδιάζεται με τη μεγαλύτερη τυπική απόκλιση (2.200) ώστε *πού*
χειρίζεται μια αξίωση να είναι ο κυρίαρχος παράγοντας. Ο σπόρος είναι
σταθεροποιημένος με `call streaminit(20260531)`. Ένας συμπαγής, πλήρως
ισορροπημένος σχεδιασμός διατηρεί κάθε επίπεδο της ιεραρχίας γεμάτο με
πραγματικούς βαθμούς ελευθερίας.


# Αποσύνθεση της Μεταβλητότητας Διακανονισμού Αξιώσεων με την PROC NESTED

Όταν μια ασφαλιστική εταιρεία περιουσίας και ατυχημάτων εξετάζει πόσο
πληρώνει για να διακανονίσει αξιώσεις αυτοκινήτου, συχνά βρίσκει μεγάλη
διακύμανση. Μέρος αυτής της διακύμανσης είναι αναπόφευκτο (κάθε
ατύχημα είναι διαφορετικό), αλλά μέρος της αντικατοπτρίζει
**οργανωτικούς** παράγοντες — μια περιφέρεια λειτουργεί πλουσιότερα από
μια άλλη, ένα υποκατάστημα είναι πιο γενναιόδωρο, ένας μεμονωμένος
πραγματογνώμονας είναι ακραία τιμή.

Τα δεδομένα έχουν αυστηρά **ένθετη** (ιεραρχική) δομή:

```
Περιφέρεια  ->  Υποκατάστημα (ένθετο εντός Περιφέρειας)  ->  Πραγματογνώμονας (ένθετος εντός Υποκαταστήματος)  ->  μεμονωμένες αξιώσεις
```

Ένα υποκατάστημα ανήκει σε ακριβώς μία περιφέρεια και ένας
πραγματογνώμονας σε ακριβώς ένα υποκατάστημα, οπότε οι παράγοντες είναι
ένθετοι και όχι διασταυρούμενοι. Η `PROC NESTED` εκτελεί μια ανάλυση
διακύμανσης τυχαίων επιδράσεων ακριβώς για αυτόν τον σχεδιασμό και
εκτιμά μια **συνιστώσα διακύμανσης** σε κάθε επίπεδο, εκφρασμένη ως
ποσοστό του συνόλου. Αυτή η ποσοστιαία κατανομή απαντά στο επιχειρηματικό
ερώτημα: *ποιο επίπεδο του οργανισμού πρέπει να στοχεύσουμε ώστε οι
διακανονισμοί να γίνουν πιο συνεπείς;*


## Βήμα 1 — Δημιουργία ενός συνθετικού, πλήρως ένθετου βιβλίου αξιώσεων

Προσομοιώνουμε 5 περιφέρειες, καθεμία με 2 υποκαταστήματα, καθένα με 2
πραγματογνώμονες, καθένας χειρίζεται 2 αξιώσεις (40 γραμμές συνολικά).
Χτίζουμε την απόκριση από προσθετικές τυχαίες επιδράσεις ώστε κάθε
επίπεδο να συνεισφέρει πραγματικά διακύμανση:

- μια επίδραση **περιφέρειας** (μεγαλύτερη διασπορά — οι περιφέρειες
  διαφέρουν περισσότερο),
- μια επίδραση **υποκαταστήματος-εντός-περιφέρειας**,
- μια επίδραση **πραγματογνώμονα-εντός-υποκαταστήματος**,
- και ένα **υπόλοιπο επιπέδου αξίωσης** (ο θόρυβος εντός πραγματογνώμονα).

Η `call streaminit` σταθεροποιεί τον σπόρο για αναπαραγωγιμότητα· κάθε
επίδραση σχεδιάζεται με `rand('NORMAL', mean, sd)`. Η επίδραση
περιφέρειας χρησιμοποιεί τη μεγαλύτερη τυπική απόκλιση, οπότε θα πρέπει
να διεκδικήσει το μεγαλύτερο μερίδιο της συνολικής διακύμανσης.
Δημιουργούμε επίσης μια δεύτερη απόκριση, `CycleDays`, που μοιράζεται
την ίδια ιεραρχία ώστε να μπορούμε να παρουσιάσουμε αργότερα την
ανάλυση πολλαπλών αποκρίσεων.


In [1]:
ΔΕΔΟΜΕΝΑ claims;
   CALL streaminit(20260531);
   LENGTH Region $6 Branch $10 Adjuster $14;

   /* Τυχαία επίδραση επιπέδου Περιφέρειας: οι περιφέρειες διαφέρουν περισσότερο. */
   ΕΠΑΝΑΛΗΨΗ r = 1 ΕΩΣ 5;
      Region = SCAN('ΒΑ ΝΑ ΜΔ ΔΑ ΝΔ', r);
      RegionEffect  = rand('NORMAL', 0, 2200);
      RegionCycle   = rand('NORMAL', 0, 11);

      /* Υποκατάστημα ένθετο εντός περιφέρειας. */
      ΕΠΑΝΑΛΗΨΗ b = 1 ΕΩΣ 2;
         Branch = catx('-', Region, PUT(b, z2.));
         BranchEffect = rand('NORMAL', 0, 700);
         BranchCycle  = rand('NORMAL', 0, 4);

         /* Πραγματογνώμονας ένθετος εντός υποκαταστήματος. */
         ΕΠΑΝΑΛΗΨΗ a = 1 ΕΩΣ 2;
            Adjuster = catx('-', Branch, PUT(a, z1.));
            AdjEffect = rand('NORMAL', 0, 450);
            AdjCycle  = rand('NORMAL', 0, 2.5);

            /* Μεμονωμένες αξιώσεις που χειρίζεται αυτός ο πραγματογνώμονας. */
            ΕΠΑΝΑΛΗΨΗ claim = 1 ΕΩΣ 2;
               Settlement = 8500
                          + RegionEffect + BranchEffect + AdjEffect
                          + rand('NORMAL', 0, 1100);
               CycleDays  = 21
                          + RegionCycle + BranchCycle + AdjCycle
                          + rand('NORMAL', 0, 6);
               ΕΑΝ Settlement < 0 ΤΟΤΕ Settlement = 0;
               ΕΑΝ CycleDays  < 1 ΤΟΤΕ CycleDays  = 1;
               ΕΞΟΔΟΣ;
            ΤΕΛΟΣ;
         ΤΕΛΟΣ;
      ΤΕΛΟΣ;
   ΤΕΛΟΣ;

   ΚΡΑΤΗΣΗ Region Branch Adjuster Settlement CycleDays;
ΕΚΤΕΛΕΣΗ;



NOTE: DATA claims


NOTE: Wrote claims (40 rows, 5 columns).
NOTE: DATA elapsed:
  wall  0.00 seconds
  cpu   0.00 seconds


## Βήμα 2 — Ταξινόμηση κατά την ένθετη ιεραρχία

Η `PROC NESTED` απαιτεί τα δεδομένα εισόδου να είναι **ταξινομημένα κατά
τις μεταβλητές CLASS με τη σειρά που θα αναφερθούν** — πρώτα ο πιο
εξωτερικός παράγοντας. Ταξινομούμε κατά `Region Branch Adjuster` ώστε
η διαδικασία να μπορεί να διατρέξει σωστά την ιεραρχία.


In [2]:
ΔΙΑΔΙΚΑΣΙΑ SORT ΔΕΔΟΜΕΝΑ=claims;
   ΚΑΤΑ Region Branch Adjuster;
ΕΚΤΕΛΕΣΗ;



NOTE: PROC SORT data=claims

NOTE: Unlicensed mode - input limited to 100 observations.
NOTE: Read 40 rows from claims.
NOTE: Wrote claims (40 rows, 5 columns).
NOTE: PROC SORT statement used.


## Βήμα 3 — Ανάλυση συνιστωσών διακύμανσης του ποσού διακανονισμού

Η βασική ανάλυση. Αναφέρουμε τις μεταβλητές CLASS από το πιο εξωτερικό
προς το πιο εσωτερικό (`Region Branch Adjuster`)· η εσωτερικότερη
επανάληψη — οι μεμονωμένες αξιώσεις — αντιμετωπίζεται αυτόματα ως ο
όρος σφάλματος. Η `VAR Settlement` ονομάζει τη μοναδική απόκριση.

Η δήλωση `LABEL` παρέχει ένα πιο φιλικό όνομα εμφάνισης για την
απόκριση στην κεφαλίδα εξόδου. Η έξοδος παράγει τους συντελεστές των
αναμενόμενων μέσων τετραγώνων, τον πίνακα ανάλυσης διακύμανσης (με
έλεγχο F κάθε συνιστώσας διακύμανσης έναντι του μηδενός), και την
εκτίμηση της **συνιστώσας διακύμανσης** σε κάθε επίπεδο μαζί με το
**ποσοστό του συνόλου**.


In [3]:
TITLE1 'Συνιστώσες Διακύμανσης των Διακανονισμών Αξιώσεων Αυτοκινήτου';
TITLE2 'ANOVA Τυχαίων Επιδράσεων Περιφέρειας / Υποκαταστήματος / Πραγματογνώμονα';

ΔΙΑΔΙΚΑΣΙΑ nested ΔΕΔΟΜΕΝΑ=claims;
   ΚΛΑΣΗ Region Branch Adjuster;
   ΜΕΤΑΒΛΗΤΗ Settlement;
   ΕΤΙΚΕΤΑ Region = 'Περιφέρεια'
         Branch = 'Υποκατάστημα'
         Adjuster = 'Πραγματογνώμονας'
         Settlement = 'Αποζημίωση Καταβληθείσα (USD)';
ΕΚΤΕΛΕΣΗ;


                             Συνιστώσες Διακύμανσης των Διακανονισμών Αξιώσεων Αυτοκινήτου                              
                        ANOVA Τυχαίων Επιδράσεων Περιφέρειας / Υποκαταστήματος / Πραγματογνώμονα                        


                       Nested Random Effects Analysis of Variance

Nested Random Effects Analysis of Variance for Variable Αποζημίωση Καταβληθείσα (USD)
Variance Source       DF  Sum of Squares       F Value      Pr > F  Error Term   Mean Square  Variance Component    Percent of Total    EMS Coef
Περιφέρεια             4  62114122.126866          6.71      0.0304  Υποκατάστημα  15528530.531717  1651824.844989             54.4057      8.0000
Υποκατάστημα           5  11569658.859028          1.89      0.1829  Πραγματογνώμονας  2313931.771806   272682.828942              8.9813      4.0000
Πραγματογνώμονας      10  12232004.560376          1.22      0.3348     Error  1223200.456038   111582.782346              3.6752      2.0000
Error              


NOTE: Option TITLE changed to Συνιστώσες Διακύμανσης των Διακανονισμών Αξιώσεων Αυτοκινήτου.
NOTE: Option TITLE2 changed to ANOVA Τυχαίων Επιδράσεων Περιφέρειας / Υποκαταστήματος / Πραγματογνώμονα.
NOTE: PROC NESTED nested ANOVA / variance components

NOTE: PROC NESTED statement used.


## Βήμα 4 — Ανάλυση δύο αποκρίσεων ταυτόχρονα

Η διοίκηση ενδιαφέρεται επίσης για τον **χρόνο κύκλου** — πόσο καιρό
χρειάζονται οι αξιώσεις για να διακανονιστούν. Προσθέτουμε το
`CycleDays` στη λίστα `VAR`. Με περισσότερες από μία εξαρτημένες
μεταβλητές, η `PROC NESTED` αναφέρει επιπλέον μια **ανάλυση
συνδιακύμανσης** που δείχνει πώς οι δύο αποκρίσεις συμμεταβάλλονται σε
κάθε επίπεδο της ιεραρχίας (για παράδειγμα, αν οι περιφέρειες που
πληρώνουν περισσότερο διακανονίζουν επίσης πιο αργά).


In [4]:
TITLE1 'Συνιστώσες Διακύμανσης Διακανονισμού και Χρόνου Κύκλου';
TITLE2 'Δύο Αποκρίσεις στην Ίδια Ένθετη Ιεραρχία';

ΔΙΑΔΙΚΑΣΙΑ nested ΔΕΔΟΜΕΝΑ=claims;
   ΚΛΑΣΗ Region Branch Adjuster;
   ΜΕΤΑΒΛΗΤΗ Settlement CycleDays;
   ΕΤΙΚΕΤΑ Region = 'Περιφέρεια'
         Branch = 'Υποκατάστημα'
         Adjuster = 'Πραγματογνώμονας'
         Settlement = 'Αποζημίωση Καταβληθείσα (USD)'
         CycleDays  = 'Ημέρες μέχρι τον Διακανονισμό';
ΕΚΤΕΛΕΣΗ;


                                 Συνιστώσες Διακύμανσης Διακανονισμού και Χρόνου Κύκλου                                 
                                        Δύο Αποκρίσεις στην Ίδια Ένθετη Ιεραρχία                                        


                       Nested Random Effects Analysis of Variance

Nested Random Effects Analysis of Variance for Variable Αποζημίωση Καταβληθείσα (USD)
Variance Source       DF  Sum of Squares       F Value      Pr > F  Error Term   Mean Square  Variance Component    Percent of Total    EMS Coef
Περιφέρεια             4  62114122.126866          6.71      0.0304  Υποκατάστημα  15528530.531717  1651824.844989             54.4057      8.0000
Υποκατάστημα           5  11569658.859028          1.89      0.1829  Πραγματογνώμονας  2313931.771806   272682.828942              8.9813      4.0000
Πραγματογνώμονας      10  12232004.560376          1.22      0.3348     Error  1223200.456038   111582.782346              3.6752      2.0000
Error              


NOTE: Option TITLE changed to Συνιστώσες Διακύμανσης Διακανονισμού και Χρόνου Κύκλου.
NOTE: Option TITLE2 changed to Δύο Αποκρίσεις στην Ίδια Ένθετη Ιεραρχία.
NOTE: PROC NESTED nested ANOVA / variance components

NOTE: PROC NESTED statement used.


## Βήμα 5 — Προβολή μόνο διακύμανσης με την επιλογή AOV

Για μια γρήγορη σύνοψη συνιστωσών διακύμανσης και για τις δύο
αποκρίσεις, η επιλογή `AOV` περιορίζει την έξοδο στα στατιστικά
ανάλυσης διακύμανσης και **καταστέλλει την ενότητα ανάλυσης
συνδιακύμανσης**. Αυτή είναι η συμπαγής προβολή που θα εξέταζε ένας
αναλυτής χαρτοφυλακίου όταν χρειάζεται μόνο την κατανομή διακύμανσης
ανά επίπεδο για κάθε απόκριση, όχι τη συνδιακύμανση μεταξύ αποκρίσεων.


In [5]:
TITLE1 'Μόνο Συνιστώσες Διακύμανσης (AOV)';

ΔΙΑΔΙΚΑΣΙΑ nested ΔΕΔΟΜΕΝΑ=claims aov;
   ΚΛΑΣΗ Region Branch Adjuster;
   ΜΕΤΑΒΛΗΤΗ Settlement CycleDays;
   ΕΤΙΚΕΤΑ Region = 'Περιφέρεια'
         Branch = 'Υποκατάστημα'
         Adjuster = 'Πραγματογνώμονας'
         Settlement = 'Αποζημίωση Καταβληθείσα (USD)'
         CycleDays  = 'Ημέρες μέχρι τον Διακανονισμό';
ΕΚΤΕΛΕΣΗ;

TITLE;


                                           Μόνο Συνιστώσες Διακύμανσης (AOV)                                            
                                        Δύο Αποκρίσεις στην Ίδια Ένθετη Ιεραρχία                                        


                       Nested Random Effects Analysis of Variance

Nested Random Effects Analysis of Variance for Variable Αποζημίωση Καταβληθείσα (USD)
Variance Source       DF  Sum of Squares       F Value      Pr > F  Error Term   Mean Square  Variance Component    Percent of Total    EMS Coef
Περιφέρεια             4  62114122.126866          6.71      0.0304  Υποκατάστημα  15528530.531717  1651824.844989             54.4057      8.0000
Υποκατάστημα           5  11569658.859028          1.89      0.1829  Πραγματογνώμονας  2313931.771806   272682.828942              8.9813      4.0000
Πραγματογνώμονας      10  12232004.560376          1.22      0.3348     Error  1223200.456038   111582.782346              3.6752      2.0000
Error              


NOTE: Option TITLE changed to Μόνο Συνιστώσες Διακύμανσης (AOV).
NOTE: PROC NESTED nested ANOVA / variance components

NOTE: PROC NESTED statement used.


## Ερμηνεία των αποτελεσμάτων

Η στήλη **Percent of Total** στον πίνακα ανάλυσης διακύμανσης είναι το
βασικό εύρημα. Διαβάζοντάς την από πάνω προς τα κάτω δίνει το μερίδιο
της συνολικής μεταβλητότητας διακανονισμού που αποδίδεται σε κάθε
οργανωτικό επίπεδο. Για το ποσό διακανονισμού η εκτέλεση παράγει:

- **Περιφέρεια — 54,4%** (Pr > F = 0,0304). Τα δεδομένα δημιουργήθηκαν
  με τη μεγαλύτερη διασπορά σε επίπεδο περιφέρειας, και η ανάλυση την
  ανακτά: περισσότερο από το μισό της συνολικής μεταβλητότητας
  διακανονισμού βρίσκεται *μεταξύ* περιφερειών, στατιστικά σημαντική
  απόδειξη ότι *πού* χειρίζεται μια αξίωση είναι ο κυρίαρχος παράγοντας.
- **Υποκατάστημα (εντός Περιφέρειας) — 9,0%** (Pr > F = 0,18). Ένα
  μέτριο επιπλέον μερίδιο μόλις κατεβούμε από την περιφέρεια στα
  μεμονωμένα υποκαταστήματα· δεν είναι στατιστικά σημαντικό εδώ.
- **Πραγματογνώμονας (εντός Υποκαταστήματος) — 3,7%** (Pr > F = 0,33).
  Ελάχιστη διακύμανση ανιχνεύεται στον μεμονωμένο πραγματογνώμονα σε
  αυτό το βιβλίο εργασιών.
- **Σφάλμα — 32,9%.** Ο υπολειπόμενος θόρυβος από αξίωση σε αξίωση
  εντός ενός πραγματογνώμονα· αυτή είναι η μη αναγώγιμη διακύμανση που
  κανένας οργανωτικός μοχλός δεν μπορεί να αφαιρέσει.

Κάθε επίπεδο φέρει επίσης έναν **έλεγχο F (Pr > F)** της μηδενικής
υπόθεσης ότι η συνιστώσα διακύμανσής του είναι μηδέν. Η μικρή τιμή p
της Περιφέρειας (0,0304) είναι στατιστική απόδειξη γνήσιων συστηματικών
διαφορών μεταξύ περιφερειών, όχι τυχαιότητας δειγματοληψίας.

Η απόκριση χρόνου κύκλου λέει μια παράλληλη ιστορία: η **Περιφέρεια
αντιστοιχεί στο 45,8%** της διακύμανσης στις ημέρες-μέχρι-διακανονισμό
(Pr > F = 0,0448, και πάλι σημαντικό), με το Υποκατάστημα και τον
Πραγματογνώμονα να συνεισφέρουν μονοψήφια μερίδια και το υπόλοιπο να
φέρει 40,1%. Άρα τόσο *πόσο* πληρώνει μια ασφαλιστική όσο και *πόσο
καιρό* χρειάζεται καθορίζονται πρώτα απ' όλα από την περιφέρεια.

Η εκτέλεση με τις δύο αποκρίσεις προσθέτει μια **ανάλυση
συνδιακύμανσης**. Εδώ το επίπεδο Περιφέρειας οδηγεί τα γινόμενα
συνδιακύμανσης, και ο συνολικός **συντελεστής συσχέτισης είναι -0,36**
— μια αρνητική σχέση: οι περιφέρειες που πληρώνουν μεγαλύτερους
διακανονισμούς τείνουν να τους *κλείνουν πιο γρήγορα*, όχι πιο αργά.
Αυτό είναι ένα χρήσιμο, μη προφανές εύρημα — οι ακριβές περιφέρειες δεν
είναι οι πιο αργές.

**Επιχειρηματικό συμπέρασμα:** επειδή η Περιφέρεια κυριαρχεί στην
κατανομή ποσοστού-του-συνόλου και για τις δύο αποκρίσεις, η ασφαλιστική
πρέπει να τυποποιήσει τις κατευθυντήριες γραμμές διακανονισμού και τα
όρια εξουσιοδότησης *μεταξύ περιφερειών* πρώτα — εκεί βρίσκεται η
μεγαλύτερη, στατιστικά σημαντική ασυνέπεια — πριν επενδύσει σε
παρεμβάσεις σε επίπεδο υποκαταστήματος ή πραγματογνώμονα. Η αρνητική
συσχέτιση διακανονισμού/χρόνου-κύκλου σημαίνει ότι δεν υπάρχει καμία
περιφέρεια που να είναι ταυτόχρονα η πιο ακριβή και η πιο αργή, οπότε
τα δύο προβλήματα μπορούν να αντιμετωπιστούν με ξεχωριστά, στοχευμένα
ανά περιφέρεια προγράμματα. Η PROC NESTED μετατρέπει μια αόριστη
αίσθηση ότι "οι διακανονισμοί μας είναι ασυνεπείς" σε μια αξιοποιήσιμη,
επίπεδο-προς-επίπεδο απόδοση του πού βρίσκεται αυτή η ασυνέπεια.
